In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_DIR = PROJECT_ROOT
CPCB_DIR = DATA_ROOT / "cpcb"
OUTPUT_DIR = OUTPUT_ROOT / "cpcb"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AOI_PATH = DATA_ROOT / "boundaries" / "ganga_kanpur_varanasi.shp"

print(f"CPCB input directory: {CPCB_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
csv_files = sorted(CPCB_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files:\n")

for file in csv_files:
    print(file.name)

merge all csv

In [ ]:
dfs = []

for file in csv_files:
    print(f"Loading {file.name}...")

    df = pd.read_csv(file)

    df["Source_File"] = file.stem

    dfs.append(df)

cpcb = pd.concat(dfs, ignore_index=True)

print("\nMerged Shape :", cpcb.shape)

In [ ]:
print("Columns :", len(cpcb.columns))

cpcb.head()

In [ ]:
cpcb.columns.tolist()

In [ ]:
# All water quality parameter columns
meta_cols = [
    "State",
    "District",
    "Basin",
    "Sub_Basin",
    "Station_Name",
    "Station_Type",
    "Date",
    "Month",
    "Year",
    "Agency_name",
    "Latitude",
    "Longitude",
    "Source_File"
]

parameter_cols = [c for c in cpcb.columns if c not in meta_cols]

print(f"Metadata Columns : {len(meta_cols)}")
print(f"Parameter Columns: {len(parameter_cols)}")

parameter_cols

In [ ]:
availability = (
    cpcb[parameter_cols]
    .notna()
    .sum()
    .sort_values(ascending=False)
)

availability.head(50)

In [ ]:
availability.to_frame("Available Samples")

In [ ]:
# Parameters we are interested in

target_cols = [
    "ph_gen",
    "ec_gen",
    "_do",
    "bod",
    "cod",
    "temp",
    "no2_n",
    "no3_n",
    "nh3_n"
]

cpcb[target_cols].head(10)

In [ ]:
cpcb[target_cols].describe(include="all")

In [ ]:
cpcb[target_cols].dtypes

In [ ]:
print("Unique Districts:")
print(sorted(cpcb["District"].dropna().unique()))

print("\nUnique Station Count:")
print(cpcb["Station_Name"].nunique())

print("\nTop 30 Stations:")
print(cpcb["Station_Name"].value_counts().head(30))

In [ ]:
print("Year Range:")
print(sorted(cpcb["Year"].dropna().unique()))

finding data from 2015-2025

In [ ]:
print(cpcb["Year"].min())
print(cpcb["Year"].max())

In [ ]:
cpcb_2015 = cpcb[
    (cpcb["Year"] >= 2015) &
    (cpcb["Year"] <= 2025)
].copy()

print(cpcb_2015.shape)

In [ ]:
print("Stations :", cpcb_2015["Station_Name"].nunique())

print("Years :", sorted(cpcb_2015["Year"].unique()))

In [ ]:
print(cpcb_2015["Station_Name"].value_counts())

findig only ganga river

In [ ]:
station_master = (
    cpcb_2015[
        ["Station_Name", "Latitude", "Longitude"]
    ]
    .drop_duplicates()
    .sort_values("Station_Name")
    .reset_index(drop=True)
)

station_master

In [ ]:
station_master.to_csv(
    OUTPUT_DIR / "Station_Master.csv",
    index=False
)

print(station_master.shape)

In [ ]:
station_master.head(40)

GETTING STATIONS ON SHPFILE

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

In [ ]:
AOI = gpd.read_file(AOI_PATH)

AOI = AOI.to_crs("EPSG:4326")

In [ ]:
stations_gdf = gpd.GeoDataFrame(
    station_master,
    geometry=gpd.points_from_xy(
        station_master.Longitude,
        station_master.Latitude
    ),
    crs="EPSG:4326"
)

In [ ]:
stations_inside = gpd.sjoin(
    stations_gdf,
    AOI,
    how="inner",
    predicate="intersects"
)

print(stations_inside.shape)

stations_inside[["Station_Name", "Latitude", "Longitude"]]

In [ ]:
print(AOI.crs)
print(stations_gdf.crs)

In [ ]:
print(AOI.total_bounds)

In [ ]:
print(stations_gdf.total_bounds)

In [ ]:
print(AOI.geom_type.unique())

In [ ]:
AOI.plot()

river centerline ka buffer

In [ ]:
import geopandas as gpd

# Convert to metric CRS (UTM Zone 44N)
AOI_buffer = AOI.to_crs(epsg=32644)

# Create 1000 m buffer around river
AOI_buffer["geometry"] = AOI_buffer.buffer(1000)

# Convert back to WGS84
AOI_buffer = AOI_buffer.to_crs(epsg=4326)

In [ ]:
stations_inside = gpd.sjoin(
    stations_gdf,
    AOI_buffer,
    how="inner",
    predicate="intersects"
)

print("Stations inside buffer:", stations_inside.shape)

stations_inside[["Station_Name", "Latitude", "Longitude"]]

In [ ]:
AOI_buffer["geometry"] = AOI_buffer.buffer(500)

FILTERING GANGA STATIONS

In [ ]:
keep_keywords = [
    "GANGA",
    "KANPUR",
    "JAJMAU",
    "BHITHAURA",
    "BHITAURA",
    "ALLAHABAD",
    "PRAYAGRAJ",
    "MIRZAPUR",
    "CHUNAR",
    "VARANASI",
    "PARMARTH",
    "SHASTRI"
]

remove_keywords = [
    "VARUNA",
    "YAMUNA",
    "TONS",
    "GOMTI",
    "KALI"
]

stations_final = stations_inside.copy()

stations_final = stations_final[
    stations_final["Station_Name"].str.upper().apply(
        lambda x: any(k in x for k in keep_keywords)
    )
]

stations_final = stations_final[
    ~stations_final["Station_Name"].str.upper().apply(
        lambda x: any(k in x for k in remove_keywords)
    )
]

stations_final[["Station_Name"]]

In [ ]:
print(len(stations_final))
print(stations_final["Station_Name"].tolist())

In [ ]:
selected_station_names = stations_final["Station_Name"].unique()

cpcb_selected = cpcb_2015[
    cpcb_2015["Station_Name"].isin(selected_station_names)
].copy()

print(cpcb_selected.shape)
print(cpcb_selected["Station_Name"].nunique())

In [ ]:
cpcb_selected.to_csv(
    OUTPUT_DIR / "CPCB_Selected_Stations_2015_2025.csv",
    index=False
)

print("Saved successfully!")

In [ ]:
station_counts = (
    cpcb_selected
    .groupby("Station_Name")
    .size()
    .sort_values(ascending=False)
)

station_counts

In [ ]:
cpcb_selected[["Date", "Year", "Month"]].head(20)

Monitoring Stations load 

In [ ]:
from pathlib import Path
import pandas as pd

monitoring = pd.read_csv(
    OUTPUT_ROOT / "monitoring_stations" / "Monitoring_Stations.csv"
)

monitoring.head()

In [ ]:
monitoring.columns

CPCB stations ko nearest monitoring station assign

In [ ]:
from scipy.spatial import cKDTree
import numpy as np

# CPCB coordinates
cpcb_coords = np.radians(
    stations_final[["Latitude", "Longitude"]].values
)

# Monitoring point coordinates
monitor_coords = np.radians(
    monitoring[["Latitude", "Longitude"]].values
)

# KDTree
tree = cKDTree(monitor_coords)

# Find nearest monitoring point
distance, index = tree.query(cpcb_coords, k=1)

# Copy dataframe
station_mapping = stations_final.copy()

# Assign nearest Station_ID
station_mapping["Station_ID"] = monitoring.iloc[index]["Station_ID"].values

# Approximate distance in meters
EARTH_RADIUS = 6371000

station_mapping["Distance_m"] = distance * EARTH_RADIUS

station_mapping[
    [
        "Station_Name",
        "Station_ID",
        "Distance_m"
    ]
]

In [ ]:
cpcb_selected = cpcb_selected.merge(
    station_mapping[
        [
            "Station_Name",
            "Station_ID",
            "Distance_m"
        ]
    ],
    on="Station_Name",
    how="left"
)

print(cpcb_selected.shape)

cpcb_selected.head()

In [ ]:
master_features = pd.read_csv(
    OUTPUT_ROOT / "features" / "Master_Features_2015_2025.csv"
)

print(master_features.shape)

master_features.head()

In [ ]:
master_features.columns.tolist()

In [ ]:
ml_dataset = cpcb_selected.merge(
    master_features,
    on=[
        "Station_ID",
        "Year",
        "Month"
    ],
    how="left"
)

print(ml_dataset.shape)

ml_dataset.head()

In [ ]:
print(ml_dataset.isna().sum())

In [ ]:
print(ml_dataset.columns)

final validation

In [ ]:
print("Rows :", len(ml_dataset))
print("Columns :", len(ml_dataset.columns))

print("\nStations :", ml_dataset["Station_Name"].nunique())

print("\nYears :")
print(sorted(ml_dataset["Year"].unique()))

print("\nMonths :")
print(sorted(ml_dataset["Month"].unique()))

In [ ]:
targets = [
    "_do",
    "bod",
    "cod",
    "ph_gen",
    "ec_gen",
    "temp"
]

ml_dataset[targets].isna().sum()

In [ ]:
feature_cols = [
    "Blue",
    "Green",
    "Red",
    "VIIRS",
    "LULC"
]

ml_dataset[feature_cols].isna().sum()

In [ ]:
ml_dataset.to_csv(
    OUTPUT_DIR / "ML_Dataset_2015_2025.csv",
    index=False
)

print("ML Dataset Saved Successfully!")